# Test Case 6 — Human Performance / Procedure Gap During Startup

## Purpose

Demonstrates the **Human Performance Assessment block** (Category H/I) and the
`training_records` input pathway on a realistic post-maintenance startup failure.

## Scenario Summary

| Field | Value |
|---|---|
| **Event ID** | `E2026-01-22-001` |
| **System** | U2 Main Feedwater Pump B |
| **Failure** | Bearing temp trip 18s after restart — lube oil venting step skipped |
| **Human factors** | Schedule pressure → supervisor accepted incomplete WO; procedure has no testable AC |
| **Primary hypothesis** | `FM-MFPB-LUBE-OIL-OMISSION` (Category H: execution error) |

## Show-stopper

The pipeline surfaces **two independent programmatic findings** from one event:
1. `human_performance_assessment` → `execution_error` (the skipped step)
2. `ishikawa_matrix[process_procedure]` → no quantitative acceptance criterion

Training records show the technician is **fully qualified** — the pipeline confirms
training is NOT the root cause by deprioritising any training-gap hypothesis. A manager
sees the correct story immediately: supervision failure under schedule pressure, plus a
procedure that cannot be verified as complete.

In [ ]:
from __future__ import annotations
import json, os, sys
from pathlib import Path

NOTEBOOK_ROOT = Path.cwd().resolve()
FIXTURE_DIR   = NOTEBOOK_ROOT / "fixtures"
OUTPUT_DIR    = NOTEBOOK_ROOT / "rca_runs_case_006"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [
    os.path.abspath(os.path.join(os.getcwd(), "..", ".."))        ,  # RCA root
    os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")) ,  # dackar root
    os.path.abspath(os.path.join(os.getcwd(), "..", "shared"))    ,  # shared helpers
]:
    if p not in sys.path:
        sys.path.insert(0, p)

from run_helpers import build_fixture_orchestrator, load_fixtures, run_rca, summarise_result
from assertion_helpers import (
    assert_candidate_present, assert_composite_score_above,
    assert_data_coverage_status, assert_human_perf_applicable,
    assert_human_perf_mode_present, assert_ishikawa_category_present,
    assert_ap913_completeness_present,
    run_assertion_table,
)
print("Imports OK. Fixture dir:", FIXTURE_DIR)

In [ ]:
fixtures = load_fixtures(FIXTURE_DIR)
print("Fixtures loaded:")
for k, v in fixtures.items():
    print(f"  {k}: {'present' if v is not None else 'absent'}")
print()
print("Training records:", len((fixtures.get('training_records') or {}).get('records', [])), "records")

In [ ]:
orchestrator = build_fixture_orchestrator(OUTPUT_DIR, top_k_candidates=5, enable_ishikawa=True)
result = run_rca(orchestrator, fixtures)
print("Pipeline complete.")
summarise_result(result)

In [ ]:
## Inspect Human Performance Assessment block
rca_card = (result.get("run_manifest") or {}).get("artifacts", {}).get("rca_card") or \
           result.get("rca_card") or {}
hp = rca_card.get("human_performance_assessment") or {}
print("=== Human Performance Assessment ===")
print(f"  applicable : {hp.get('applicable')}")
findings = hp.get("findings") or []
for i, f in enumerate(findings):
    print(f"  finding[{i}] : mode={f.get('performance_mode')}  label={f.get('label')}")

## Ishikawa — process/procedure row
ishikawa = result.get("ishikawa_matrix") or {}
proc_rows = ishikawa.get("process_procedure") or []
print(f"\n=== Ishikawa — process_procedure rows: {len(proc_rows)} ===")
for row in proc_rows:
    print(f"  {row}")

In [ ]:
## Training records — confirm training currency and that it is NOT the root cause
coverage = ((result.get("run_manifest") or {}).get("coverage_summary") or {}).get("source_families") or {}
tr_entry = coverage.get("training_records")
tr_status = tr_entry.get("status") if isinstance(tr_entry, dict) else tr_entry
print(f"Training records coverage status: {tr_status}")

candidates = (result.get("causality_candidates") or {}).get("candidates") or []
print("\nCandidate ranking:")
for c in sorted(candidates, key=lambda x: -float(x.get("composite_score", 0) or 0)):
    scores = c.get("scores") or {}
    print(f"  [{c.get('failure_mode_id')}]  composite={c.get('composite_score', '?')}  cat={c.get('primary_causal_category', '?')}  op_pt={scores.get('operating_point_score', '?')}")

In [ ]:
assertions = [
    {"id": "A6-1", "desc": "Primary hypothesis is execution error",
     "fn": lambda r: assert_candidate_present(r, "FM-MFPB-LUBE-OIL-OMISSION")},
    {"id": "A6-2", "desc": "Human performance assessment applicable",
     "fn": lambda r: assert_human_perf_applicable(r)},
    {"id": "A6-3", "desc": "Execution error finding present",
     "fn": lambda r: assert_human_perf_mode_present(r, "execution_error")},
    {"id": "A6-4", "desc": "Process/procedure Ishikawa row present",
     "fn": lambda r: assert_ishikawa_category_present(r, "process_procedure")},
    {"id": "A6-5", "desc": "Training records coverage complete",
     "fn": lambda r: assert_data_coverage_status(r, "training_records", "complete")},
    {"id": "A6-6", "desc": "AP-913 completeness block present",
     "fn": lambda r: assert_ap913_completeness_present(r)},
]
run_assertion_table(result, assertions, label="TC-6 Assertions")

In [ ]:
out_path = OUTPUT_DIR / "tc6_full_result.json"
with open(out_path, "w", encoding="utf-8") as fh:
    json.dump(result, fh, indent=2, default=str)
print(f"Saved: {out_path}")